# Eleições presidenciais — ciência de dados

1. **Transferência de votos** do 1º para o 2º turno, estimada por seção eleitoral (inferência ecológica).
2. **Análise espacial**: o voto forma blocos geográficos? (Moran's I e LISA)
3. **Voto x perfil socioeconômico**: regressão ponderada e gradient boosting com SHAP.
4. **Comparação com a eleição anterior** (swing por município).

Cada seção roda só quando os dados necessários existem; senão, explica o que falta.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from eleicoes import graficos as gr
from eleicoes import resultados as r
from eleicoes.config import config
from eleicoes.dados import (
    ORDEM_REGIOES,
    carregar_comparecimento,
    carregar_malha,
    carregar_votos,
    municipios_ibge,
)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
%matplotlib inline

ANO = int(config()["ano_principal"])
df = carregar_votos(ANO)
comp = carregar_comparecimento(ANO)  # None se o detalhe da apuração não foi processado
TURNOS = r.turnos(df)
fin = r.finalistas(df)
NOME_A, NOME_B = r.rotulos(fin)
print(
    f"Eleição {ANO} | turnos disponíveis: {TURNOS} | finalistas: {NOME_A} ({fin['A']}) e {NOME_B} ({fin['B']})"
)
print("Abstenção:", "disponível" if comp is not None else "baixe e processe o detalhe da apuração")

from eleicoes import espacial, modelo
from eleicoes import transferencia as tr
from eleicoes.dados import base_socioeconomica, carregar_secao, disponivel

malha = carregar_malha()
ULTIMO = TURNOS[-1]
MUN = {t: r.municipios(df, t, fin) for t in TURNOS}

## 1. Para onde foram os votos? (inferência ecológica por seção)

Para cada seção, as proporções do 2º turno são modeladas como combinação das do 1º:
`y = x · P`, com `P ≥ 0` e linhas somando 100%. `P` é estimada por UF e combinada em uma
matriz nacional; o intervalo de confiança vem de bootstrap por município.

**Como ler:** a linha "Candidato X" diz, de cada 100 eleitores de X no 1º turno, quantos
foram para cada opção no 2º. **Limites:** é uma estimativa de padrão médio a partir de
agregados (não do voto individual), e quem deixou de votar entre os turnos não é modelado.
Leva de 1 a 3 minutos na base completa.

In [ ]:
if 2 not in TURNOS:
    print("Precisa do 2º turno nos dados.")
else:
    MENORES = r.candidatos_menores(df, fin, minimo_pct=1.0)
    X, Y = tr.preparar(carregar_secao(ANO), fin, MENORES)
    T = tr.estimar(X, Y, n_boot=200)
    gr.matriz_transferencia(T, "Para onde foram os votos do 1º turno")
    plt.show()
    display(tr.tabela(T))

In [ ]:
if 2 in TURNOS:
    # Como os eleitores do principal candidato eliminado se dividiram em cada UF
    if MENORES:
        terceiro = fin["rotulos"][MENORES[0]]
        por_uf = pd.DataFrame({uf: P.loc[terceiro] * 100 for uf, P in T.por_uf.items()}).T
        display(por_uf.sort_values(NOME_A, ascending=False).round(1))

## 2. O voto forma blocos geográficos?

**Moran's I** perto de 0 = distribuição aleatória; positivo = vizinhos votam parecido.
**LISA** mostra onde estão os blocos: municípios fortes para um candidato cercados de
vizinhos também fortes, e as exceções que destoam da vizinhança.

In [ ]:
g = malha.merge(MUN[ULTIMO], on="CD_MUNICIPIO", how="inner")
mg = espacial.moran(g, "dif_A_B")
print(
    f"Moran's I = {mg['I']:.3f}  (p = {mg['p_valor']:.3f}, esperado sem padrão = {mg['esperado']:.4f}, n = {mg['n']})"
)
gl = espacial.lisa(g, "dif_A_B")
gr.mapa_lisa(gl, fin, f"Blocos geográficos de voto — {ULTIMO}º turno", moran_global=mg)
plt.show()
display(
    gl.groupby("lisa", observed=False).agg(
        municipios=("CD_MUNICIPIO", "count"), votos_validos=("validos", "sum")
    )
)

## 3. Voto x perfil socioeconômico

Alvo: % dos válidos do finalista A em cada município. Indicadores: Atlas do Desenvolvimento
Humano (Censo 2010) e população/densidade (Censo 2022).

- **Regressão ponderada (WLS)**: cada coeficiente é o efeito, em pontos percentuais, de
  +1 desvio-padrão na variável, mantendo as demais e a região constantes. Erros-padrão robustos.
- **Gradient boosting + SHAP**: captura não linearidades. Validação cruzada por UF (o modelo é
  testado em estados que não viu). SHAP mostra quanto cada variável empurra a previsão.

Associação não é causalidade, e não descreve o eleitor individual.

In [ ]:
socio = base_socioeconomica()
base = modelo.montar_base(MUN[ULTIMO], socio, alvo="pct_A")
print(f"{len(base):,} municípios na base".replace(",", "."))
cols = list(modelo.VARIAVEIS_LINEAR)
display(
    base[["alvo", *cols]]
    .corr()["alvo"]
    .drop("alvo")
    .rename(modelo.VARIAVEIS_LINEAR)
    .sort_values()
    .to_frame("correlação com % de A")
)

In [ ]:
res, coef = modelo.regressao_linear(base)
gr.coeficientes(coef, f"% de {NOME_A}: efeito de cada variável (regressão ponderada)")
plt.show()
print(f"R² = {res.rsquared:.3f}")
display(coef.round(3))
display(modelo.vif(base).round(2).to_frame())

In [ ]:
gbm = modelo.modelo_gbm(base)
m = gbm["metricas"]
print(f"R² na validação por UF: {m['r2_validacao_por_uf']:.3f}")
print(f"Erro médio: {m['mae_pp']:.1f} p.p.  (chute pela média: {m['mae_baseline_pp']:.1f} p.p.)")
gr.shap_resumo(gbm, f"O que mais explica o % de {NOME_A} por município (SHAP)")
plt.show()
display(gbm["importancia_pp"].round(2).to_frame("impacto médio (p.p.)"))

## 4. Comparação com a eleição anterior

Swing = variação, em pontos percentuais, da votação de um mesmo número de urna (partido)
entre as duas eleições, no mesmo turno. Requer a eleição anterior processada
(`python -m eleicoes.processar --ano <ano_comparacao>`).

In [ ]:
ANT = config().get("ano_comparacao")
if not ANT or not disponivel(int(ANT)):
    print(
        "Eleição de comparação não processada. Rode: python -m eleicoes.baixar && "
        f"python -m eleicoes.processar --ano {ANT}"
    )
else:
    df_ant = carregar_votos(int(ANT))
    turno_cmp = ULTIMO if ULTIMO in r.turnos(df_ant) else 1
    nominais_ant = set(df_ant.loc[df_ant["TP_VOTO"] == "Nominal", "NR_VOTAVEL"])
    for numero in [fin["A"], fin["B"]]:
        if numero not in nominais_ant:
            print(f"Número {numero} não concorreu em {ANT}.")
            continue
        sw = r.swing(df, df_ant, numero, turno_cmp)
        nome = fin["rotulos"][numero]
        total = (sw["pct_atual"] * sw["validos"]).sum() / sw["validos"].sum() - (
            sw["pct_anterior"] * sw["validos"]
        ).sum() / sw["validos"].sum()
        gr.mapa_continuo(
            malha.merge(sw, on="CD_MUNICIPIO", how="left"),
            "swing_pp",
            f"Nº {numero} ({nome}): variação de {ANT} para {ANO} — {turno_cmp}º turno",
            f"mapa_swing_{numero}",
            limite=20,
            cmap=gr.cores_numero(numero, fin),
            rotulo="← perdeu   |   variação (p.p.)   |   ganhou →",
            subtitulo=f"Variação ponderada pelo eleitorado: {'+' if total >= 0 else '−'}{gr.decimal(abs(total))} p.p.",
        )
        plt.show()
        grandes = sw[sw["validos"] >= 100_000].merge(
            MUN[ULTIMO][["CD_MUNICIPIO", "SG_UF", "NM_MUNICIPIO"]], on="CD_MUNICIPIO"
        )
        display(
            pd.concat([grandes.nlargest(8, "swing_pp"), grandes.nsmallest(8, "swing_pp")])[
                ["SG_UF", "NM_MUNICIPIO", "validos", "pct_anterior", "pct_atual", "swing_pp"]
            ]
        )